# 03 CV SPLIT

Extracted from the original notebooks. **Outputs preserved.** Originals are unmodified.


## A · The patient-grouped 5-fold definition (root of the CV protocol)


**`WPN` cell 2** — ══════════════════════════════════════════════════════════════════════  
<sub>1 output block(s) preserved</sub>


In [2]:
# ══════════════════════════════════════════════════════════════════════
# PHASE 1: unified patient-grouped folds shared by BOTH stages
# ══════════════════════════════════════════════════════════════════════
import os, numpy as np, pandas as pd
from sklearn.model_selection import StratifiedGroupKFold
D="/root/autodl-tmp/CBIS"; NFOLD=5; SEED=42

for LES,CSV in [("mass","cbis_mass_fixed.csv"),("calc","cbis_calc_fixed.csv")]:
    d=pd.read_csv(os.path.join(D,CSV))
    d["label"]=d["label"].astype(int)
    d["assessment"]=pd.to_numeric(d["assessment"],errors="coerce")
    d=d.dropna(subset=["img","msk","label"]).reset_index(drop=True)

    sc=[c for c in ["side","left or right breast"] if c in d.columns][0]
    lc=[c for c in ["lesion","abnormality id"] if c in d.columns][0]
    d["lesion_key"]=d.patient_id.astype(str)+"_"+d[sc].astype(str)+"_"+d[lc].astype(str)

    strat=d.label.astype(str)+"_"+d.assessment.isin([3,4]).astype(int).astype(str)
    d["fold"]=-1
    for k,(tr,te) in enumerate(StratifiedGroupKFold(NFOLD,shuffle=True,random_state=SEED)
                               .split(d,strat,d.patient_id.values)):
        d.loc[te,"fold"]=k

    # per-fold role: train / val / test  (val = 12% of that fold's training patients)
    for k in range(NFOLD):
        col="role_f"+str(k)
        d[col]="train"
        d.loc[d.fold==k,col]="test"
        trp=sorted(set(d.loc[d.fold!=k,"patient_id"]))
        rs=np.random.RandomState(1000+k)
        vp=set(rs.permutation(np.array(trp,dtype=object))[:max(1,int(0.12*len(trp)))])
        d.loc[(d.fold!=k)&(d.patient_id.isin(vp)),col]="val"

    out=os.path.join(D,"unified_folds_"+LES+".csv")
    d.to_csv(out,index=False)

    print("="*76); print(LES.upper()+"   n="+str(len(d))+"   patients="+str(d.patient_id.nunique())+
          "   lesions="+str(d.lesion_key.nunique())); print("="*76)
    print("  fold   n_img  n_pat  n_les  malig%   B4%   | train/val/test images")
    for k in range(NFOLD):
        q=d[d.fold==k]; r=d["role_f"+str(k)]
        print("   "+str(k)+"     "+str(len(q)).rjust(5)+"  "+str(q.patient_id.nunique()).rjust(5)+
              "  "+str(q.lesion_key.nunique()).rjust(5)+"   "+format(100*q.label.mean(),".1f").rjust(5)+
              "%  "+format(100*(q.assessment==4).mean(),".0f").rjust(3)+"%   | "+
              str((r=="train").sum())+"/"+str((r=="val").sum())+"/"+str((r=="test").sum()))

    bad=[]
    for k in range(NFOLD):
        r=d["role_f"+str(k)]
        trp=set(d.loc[r=="train","patient_id"]); vap=set(d.loc[r=="val","patient_id"]); tep=set(d.loc[r=="test","patient_id"])
        if (trp&tep) or (vap&tep) or (trp&vap): bad.append(k)
    print("  leakage check: "+("PASS - no patient appears in two roles in any fold"
                               if not bad else "FAIL in folds "+str(bad)))
    print("  saved "+os.path.basename(out))

MASS   n=1696   patients=892   lesions=1005
  fold   n_img  n_pat  n_les  malig%   B4%   | train/val/test images
   0       339    178    199    46.3%   41%   | 1202/155/339
   1       340    178    201    46.2%   41%   | 1184/172/340
   2       339    179    201    46.3%   42%   | 1192/165/339
   3       339    176    199    46.0%   41%   | 1195/162/339
   4       339    181    205    46.3%   42%   | 1203/154/339
  leakage check: PASS - no patient appears in two roles in any fold
  saved unified_folds_mass.csv
CALC   n=1866   patients=753   lesions=1042
  fold   n_img  n_pat  n_les  malig%   B4%   | train/val/test images
   0       374    149    206    36.1%   49%   | 1303/189/374
   1       373    150    209    35.9%   52%   | 1335/158/373
   2       373    152    208    36.2%   50%   | 1301/192/373
   3       373    151    210    35.9%   50%   | 1308/185/373
   4       373    151    209    35.9%   49%   | 1283/210/373
  leakage check: PASS - no patient appears in two roles in any fo

## B · 5-fold CV runs


**`BCF` cell 289** — ══════════════════════════════════════════════════════════════════════  
<sub>1 output block(s) preserved</sub>


In [2]:
# ══════════════════════════════════════════════════════════════════════
# 5-FOLD PATIENT-GROUPED CV — mass | calcification | INbreast
#   + BREAST-REGION FIX: masks intersected with actual breast tissue
#     (removes annotation area falling on black background)
#   preprocessing: CLAHE(2.0,8x8); augmentation 8x on TRAIN folds only
#   calc uses multi-task heads; mass/INbreast use shape features
# ══════════════════════════════════════════════════════════════════════
import os
os.environ["OMP_NUM_THREADS"]="4"; os.environ["HF_HUB_OFFLINE"]="1"; os.environ["TRANSFORMERS_OFFLINE"]="1"
import torch, torch.nn as nn, torch.nn.functional as F
import numpy as np, pandas as pd, cv2
from torch.utils.data import Dataset, DataLoader
from torchvision import models
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import (roc_auc_score, accuracy_score, f1_score,
                             balanced_accuracy_score, confusion_matrix, recall_score)
D="/root/autodl-tmp/CBIS"
DEV=torch.device("cuda" if torch.cuda.is_available() else "cpu")
SIZE=320; BATCH=10; MULT=8; EPOCHS=22
LR_HEAD=1e-3; LR_FT=1e-5; FREEZE=3; GAMMA=2.0; AUX_W=0.3; NFOLD=5
_clahe=cv2.createCLAHE(clipLimit=2.0,tileGridSize=(8,8))
MEAN=np.array([0.485,0.456,0.406],np.float32); STD=np.array([0.229,0.224,0.225],np.float32)

# ---------- data: CBIS + INbreast ----------
raw=pd.read_csv(os.path.join(D,"cbis_final.csv"))
C=raw[["img","msk","label","patient_id","abn_type","subtlety","density",
       "mass_margins","calc_type","calc_dist"]].copy(); C["source"]="CBIS"
itr=pd.read_csv(os.path.join(D,"inbreast_train_aug.csv")).rename(columns={"cropped_jpeg_path":"img","roi_mask_jpeg_path":"msk"})
ite=pd.read_csv(os.path.join(D,"inbreast_test.csv")).rename(columns={"cropped_jpeg_path":"img","roi_mask_jpeg_path":"msk"})
I=pd.concat([itr,ite],ignore_index=True)[["img","msk","label","patient_id"]].copy()
for c in ["abn_type","subtlety","density","mass_margins","calc_type","calc_dist"]: I[c]=np.nan
I["abn_type"]="mass"; I["source"]="INbreast"
ALL=pd.concat([C,I],ignore_index=True).dropna(subset=["img","label"]).reset_index(drop=True)
ALL["label"]=ALL["label"].astype(int)
print("CBIS mass "+str(((ALL.source=='CBIS')&(ALL.abn_type=='mass')).sum())+
      " | CBIS calc "+str(((ALL.source=='CBIS')&(ALL.abn_type=='calcification')).sum())+
      " | INbreast "+str((ALL.source=='INbreast').sum()))

# ---------- BREAST-REGION FIX ----------
def breast_region(img):
    """non-background (actual tissue) region of the crop"""
    g=cv2.GaussianBlur(img,(5,5),0)
    _,bw=cv2.threshold(g,0,255,cv2.THRESH_BINARY+cv2.THRESH_OTSU)
    bw=(bw>0).astype(np.uint8)
    if bw.mean()<0.05 or bw.mean()>0.995:            # Otsu failed -> fixed low threshold
        bw=(img>12).astype(np.uint8)
    k=cv2.getStructuringElement(cv2.MORPH_ELLIPSE,(7,7))
    bw=cv2.morphologyEx(bw,cv2.MORPH_CLOSE,k)
    return bw

def clean_mask(mask,img):
    """intersect annotation with breast tissue; drop area on black background"""
    b=breast_region(img)
    out=((mask>0)&(b>0)).astype(np.uint8)
    if out.sum()<0.15*max(mask.sum(),1): return (mask>0).astype(np.uint8)  # safety
    return out

# audit how much area the fix removes
aud=[]
for _,r in ALL.sample(min(400,len(ALL)),random_state=0).iterrows():
    im=cv2.imread(r["img"],cv2.IMREAD_GRAYSCALE); mk=cv2.imread(r["msk"],cv2.IMREAD_GRAYSCALE)
    if im is None or mk is None: continue
    im=cv2.resize(im,(SIZE,SIZE))
    m=(cv2.resize(mk,(SIZE,SIZE),interpolation=cv2.INTER_NEAREST)>127).astype(np.uint8)
    if m.sum()==0: continue
    c=clean_mask(m,im)
    aud.append(dict(src=r["source"],abn=r["abn_type"],
                    before=float(m.mean()),after=float(c.mean()),
                    removed=float((m.sum()-c.sum())/max(m.sum(),1))))
A=pd.DataFrame(aud)
print("\nBREAST-REGION FIX — fraction of mask area falling on black background:")
for (s,a),g in A.groupby(["src","abn"]):
    print("  "+str(s).ljust(9)+str(a).ljust(15)+"mean removed "+format(100*g.removed.mean(),".1f")+"%"+
          "   masks losing >10%: "+format(100*(g.removed>0.10).mean(),".0f")+"%")

# ---------- features / aux ----------
def primary(x): return "UNK" if pd.isna(x) else str(x).split("-")[0].strip().upper()
def build_aux(df,cols,topn=6):
    out={};meta={}
    for c in cols:
        if c not in df.columns or df[c].notna().sum()==0: continue
        if c in ("subtlety","density"):
            v=pd.to_numeric(df[c],errors="coerce"); v=v.where((v>=1)&(v<=5))
            codes=(v-1).fillna(-1).astype(int); n=int(v.max()) if v.notna().any() else 0
        else:
            p=df[c].map(primary); keep=p.value_counts().head(topn).index.tolist()
            p=p.where(p.isin(keep),"OTHER")
            cats=sorted([k for k in p.unique() if k!="UNK"]); mp={k:i for i,k in enumerate(cats)}
            codes=p.map(lambda z: mp.get(z,-1)).astype(int); n=len(cats)
        if n>1: out[c]=codes.values; meta[c]=n
    return out,meta

def shape_feats(m):
    m=(m>0).astype(np.uint8)
    if m.sum()<10: return np.zeros(6,np.float32)
    c,_=cv2.findContours(m,cv2.RETR_EXTERNAL,cv2.CHAIN_APPROX_SIMPLE)
    if not c: return np.zeros(6,np.float32)
    c=max(c,key=cv2.contourArea); a=cv2.contourArea(c); p=cv2.arcLength(c,True)+1e-6
    h=cv2.convexHull(c); ha=cv2.contourArea(h)+1e-6; x,y,w,hh=cv2.boundingRect(c)
    return np.array([4*np.pi*a/(p*p),a/ha,a/(w*hh+1e-6),w/(hh+1e-6),m.mean(),p/(a+1e-6)],np.float32)

class DS(Dataset):
    def __init__(s,df,auxarr,aug,mult=1,tta=0):
        s.df=df.reset_index(drop=True); s.aux=auxarr; s.aug=aug
        s.mult=mult if aug else 1; s.tta=tta; s.keys=sorted(auxarr.keys())
    def __len__(s): return len(s.df)*s.mult
    def __getitem__(s,i):
        j=i%len(s.df); r=s.df.iloc[j]; k=i//len(s.df)
        img=cv2.imread(r["img"],cv2.IMREAD_GRAYSCALE)
        if img is None: img=np.zeros((SIZE,SIZE),np.uint8)
        img=cv2.resize(img,(SIZE,SIZE))
        mk=cv2.imread(r["msk"],cv2.IMREAD_GRAYSCALE)
        mask=(cv2.resize(mk,(SIZE,SIZE),interpolation=cv2.INTER_NEAREST)>127).astype(np.uint8) if mk is not None else np.zeros((SIZE,SIZE),np.uint8)
        mask=clean_mask(mask,img)                     # <-- breast-region fix
        if s.aug and k>0:                             # augmentation: TRAIN folds only
            if   k%8==1: img=np.fliplr(img).copy(); mask=np.fliplr(mask).copy()
            elif k%8==2: img=np.flipud(img).copy(); mask=np.flipud(mask).copy()
            elif k%8==3: img=np.rot90(img,1).copy(); mask=np.rot90(mask,1).copy()
            elif k%8==4: img=np.rot90(img,2).copy(); mask=np.rot90(mask,2).copy()
            elif k%8==5: img=np.rot90(img,3).copy(); mask=np.rot90(mask,3).copy()
            elif k%8==6:
                M=cv2.getRotationMatrix2D((SIZE/2,SIZE/2),np.random.uniform(-20,20),np.random.uniform(.9,1.1))
                img=cv2.warpAffine(img,M,(SIZE,SIZE),borderMode=cv2.BORDER_REFLECT)
                mask=cv2.warpAffine(mask,M,(SIZE,SIZE),flags=cv2.INTER_NEAREST)
            elif k%8==7: img=np.clip(img.astype(np.float32)*np.random.uniform(.85,1.15),0,255).astype(np.uint8)
        if   s.tta==1: img=np.fliplr(img).copy(); mask=np.fliplr(mask).copy()
        elif s.tta==2: img=np.flipud(img).copy(); mask=np.flipud(mask).copy()
        elif s.tta==3: img=np.rot90(img,2).copy(); mask=np.rot90(mask,2).copy()
        im=_clahe.apply(img).astype(np.float32)/255.  # CLAHE preprocessing
        x=np.stack([im,im,im],0)
        x=((x.transpose(1,2,0)-MEAN)/STD).transpose(2,0,1).astype(np.float32)
        av=np.array([s.aux[k2][j] for k2 in s.keys],dtype=np.int64) if s.keys else np.zeros(0,np.int64)
        return torch.from_numpy(x), torch.from_numpy(shape_feats(mask)), torch.tensor(int(r["label"])), torch.from_numpy(av)

class Net(nn.Module):
    def __init__(s,aux_meta,use_aux,nfeat=6):
        super().__init__()
        try: dn=models.densenet121(weights=models.DenseNet121_Weights.IMAGENET1K_V1)
        except Exception: dn=models.densenet121(weights=None)
        s.b=dn.features; s.pool=nn.AdaptiveAvgPool2d(1)
        s.mlp=nn.Sequential(nn.Linear(nfeat,32),nn.ReLU(),nn.Linear(32,32),nn.ReLU())
        s.head=nn.Sequential(nn.Linear(1024+32,256),nn.ReLU(),nn.Dropout(0.5),nn.Linear(256,2))
        s.use_aux=use_aux; s.keys=sorted(aux_meta.keys())
        if use_aux:
            s.aux=nn.ModuleList([nn.Sequential(nn.Linear(1024,128),nn.ReLU(),nn.Dropout(0.3),
                                               nn.Linear(128,aux_meta[k])) for k in s.keys])
    def forward(s,x,f):
        g=s.pool(F.relu(s.b(x))).flatten(1)
        return s.head(torch.cat([g,s.mlp(f)],1)), ([h(g) for h in s.aux] if s.use_aux else [])

def youden(y,p):
    b=(0,0.5)
    for t in np.linspace(0.05,0.95,181):
        v=balanced_accuracy_score(y,(p>t).astype(int))
        if v>b[0]: b=(v,t)
    return b[1]

def cv_run(name,sub,aux_cols,use_aux):
    sub=sub.reset_index(drop=True)
    auxarr,meta=build_aux(sub,aux_cols) if use_aux else ({},{})
    y=sub.label.values; groups=sub.patient_id.values
    nf=min(NFOLD,len(set(groups)))
    sgkf=StratifiedGroupKFold(n_splits=nf,shuffle=True,random_state=42)
    print("\n"+"#"*68)
    print("### "+name.upper()+"   "+str(nf)+"-fold patient-grouped CV   n="+str(len(sub))+
          "   participants="+str(len(set(groups)))+"   multitask="+str(use_aux))
    print("#"*68)
    oof=np.zeros(len(sub)); fa=[]
    for fold,(tri,tei) in enumerate(sgkf.split(sub,y,groups),1):
        gtr=groups[tri]; uq=np.array(sorted(set(gtr)))
        rs=np.random.RandomState(fold); vg=set(rs.permutation(uq)[:max(1,int(0.12*len(uq)))])
        vm=np.array([g in vg for g in gtr]); tr_i=tri[~vm]; va_i=tri[vm]
        assert len(set(groups[tr_i])&set(groups[tei]))==0, "LEAK train/test fold "+str(fold)
        assert len(set(groups[va_i])&set(groups[tei]))==0, "LEAK val/test fold "+str(fold)
        tr=sub.iloc[tr_i]; va=sub.iloc[va_i]; te=sub.iloc[tei]
        sl=lambda ix:{k:v[ix] for k,v in auxarr.items()}
        n0=float((tr.label==0).sum()); n1=float((tr.label==1).sum())
        alpha=torch.tensor([n1/(n0+n1),n0/(n0+n1)],device=DEV)
        def focal(lo,t):
            ce=F.cross_entropy(lo.float(),t,weight=alpha,reduction="none"); pt=torch.exp(-ce)
            return ((1-pt)**GAMMA*ce).mean()
        torch.manual_seed(fold); np.random.seed(fold)
        net=Net(meta,use_aux).to(DEV)
        for p in net.b.parameters(): p.requires_grad=False
        sc=torch.amp.GradScaler()
        opt=torch.optim.AdamW([p for p in net.parameters() if p.requires_grad],lr=LR_HEAD,weight_decay=1e-3)
        tl=DataLoader(DS(tr,sl(tr_i),True,MULT),batch_size=BATCH,shuffle=True,num_workers=0)
        @torch.no_grad()
        def collect(df,ix,tta=True):
            net.eval(); reps=[0,1,2,3] if tta else [0]; tot=None; lab=None
            for t in reps:
                ld=DataLoader(DS(df,sl(ix),False,tta=t),batch_size=BATCH,shuffle=False,num_workers=0)
                ps=[];lb=[]
                for x,f,t2,a in ld:
                    x=x.to(DEV); f=f.to(DEV)
                    with torch.amp.autocast(device_type="cuda"): o,_=net(x,f)
                    ps+=list(torch.softmax(o.float(),1)[:,1].cpu().numpy()); lb+=list(t2.numpy())
                ps=np.array(ps); lab=np.array(lb); tot=ps if tot is None else tot+ps
            return lab,tot/len(reps)
        best=0.; bs=None; ni=0
        for ep in range(1,EPOCHS+1):
            if ep==FREEZE+1:
                for p in net.b.parameters(): p.requires_grad=True
                opt=torch.optim.AdamW(net.parameters(),lr=LR_FT,weight_decay=1e-3)
            net.train()
            if ep<=FREEZE: net.b.eval()
            for x,f,t2,a in tl:
                x=x.to(DEV); f=f.to(DEV); t2=t2.to(DEV); a=a.to(DEV)
                opt.zero_grad(set_to_none=True)
                with torch.amp.autocast(device_type="cuda"):
                    o,aux=net(x,f); lm=focal(o,t2)
                    la=torch.zeros((),device=DEV)
                    for h,lg in enumerate(aux): la=la+F.cross_entropy(lg.float(),a[:,h],ignore_index=-1)
                    if len(aux): la=la/len(aux)
                    loss=lm+AUX_W*la
                sc.scale(loss).backward(); sc.step(opt); sc.update()
            yv,pv=collect(va,va_i,tta=False)
            auc=roc_auc_score(yv,pv) if len(set(yv))>1 else 0
            if auc>best: best=auc; bs={k:v.cpu().clone() for k,v in net.state_dict().items()}; ni=0
            else: ni+=1
            if ni>=6: break
        net.load_state_dict({k:v.to(DEV) for k,v in bs.items()})
        yt,pt=collect(te,tei); oof[tei]=pt
        a=roc_auc_score(yt,pt); fa.append(a)
        print("  fold "+str(fold)+"  train "+str(len(tr))+"  test "+str(len(te))+"  AUC "+format(a,".4f"))
    thr=youden(y,oof); pred=(oof>thr).astype(int)
    cm=confusion_matrix(y,pred); tn,fp,fn,tp=cm.ravel()
    print("  "+"-"*62)
    print("  fold AUCs "+str([round(v,4) for v in fa]))
    print("  MEAN AUC   "+format(np.mean(fa),".4f")+" +/- "+format(np.std(fa),".4f"))
    print("  POOLED OOF "+format(roc_auc_score(y,oof),".4f")+
          "  acc "+format(accuracy_score(y,pred),".3f")+
          "  bal "+format(balanced_accuracy_score(y,pred),".3f")+
          "  F1 "+format(f1_score(y,pred),".3f"))
    print("  malignant "+str(tp)+"/"+str(tp+fn)+"   benign "+str(tn)+"/"+str(tn+fp))
    o=sub.copy(); o["prob"]=oof; o["true"]=y; o["pred"]=pred
    o.to_csv(os.path.join(D,"cv_oof_"+name+".csv"),index=False)
    return dict(dataset=name,n=len(sub),mean_auc=float(np.mean(fa)),std=float(np.std(fa)),
                pooled=float(roc_auc_score(y,oof)),acc=accuracy_score(y,pred),
                bal=balanced_accuracy_score(y,pred),f1=f1_score(y,pred),TN=tn,FP=fp,FN=fn,TP=tp)

R=[]
R.append(cv_run("cbis_mass", ALL[(ALL.source=="CBIS")&(ALL.abn_type=="mass")],
                ["subtlety","density","mass_margins"], use_aux=False))
R.append(cv_run("cbis_calc", ALL[(ALL.source=="CBIS")&(ALL.abn_type=="calcification")],
                ["subtlety","calc_type","calc_dist"], use_aux=True))
R.append(cv_run("inbreast",  ALL[ALL.source=="INbreast"], [], use_aux=False))
T=pd.DataFrame(R); T.to_csv(os.path.join(D,"cv_final_results.csv"),index=False)
print("\n"+"="*86)
print("5-FOLD PATIENT-GROUPED CROSS-VALIDATION — FINAL RESULTS")
print("="*86)
print("  dataset".ljust(14)+"n     mean AUC          pooled   acc    bal    F1     malig     benign")
for _,r in T.iterrows():
    print("  "+r.dataset.ljust(14)+str(int(r.n)).rjust(4)+"  "+format(r.mean_auc,".4f")+
          " +/- "+format(r["std"],".4f")+"   "+format(r.pooled,".4f")+"  "+
          format(r.acc,".3f")+"  "+format(r.bal,".3f")+"  "+format(r.f1,".3f")+"  "+
          str(int(r.TP))+"/"+str(int(r.TP+r.FN))+"   "+str(int(r.TN))+"/"+str(int(r.TN+r.FP)))
print("  "+"-"*82)
print("  single-split reference: mass 0.8137 | calc 0.7860 | INbreast 0.9405")
print("="*86)

CBIS mass 1696 | CBIS calc 1866 | INbreast 1200

BREAST-REGION FIX — fraction of mask area falling on black background:
  CBIS     calcification  mean removed 34.7%   masks losing >10%: 80%
  CBIS     mass           mean removed 18.0%   masks losing >10%: 74%
  INbreast mass           mean removed 22.7%   masks losing >10%: 77%

####################################################################
### CBIS_MASS   5-fold patient-grouped CV   n=1696   participants=892   multitask=False
####################################################################
  fold 1  train 1198  test 338  AUC 0.7313
  fold 2  train 1206  test 338  AUC 0.8385
  fold 3  train 1182  test 340  AUC 0.7487
  fold 4  train 1196  test 338  AUC 0.8238
  fold 5  train 1191  test 342  AUC 0.8230
  --------------------------------------------------------------
  fold AUCs [0.7313, 0.8385, 0.7487, 0.8238, 0.823]
  MEAN AUC   0.7931 +/- 0.0440
  POOLED OOF 0.7909  acc 0.725  bal 0.724  F1 0.704
  malignant 556/784   benign

## C · CV re-run with the official machinery + CV vs official comparison


**`IL2` cell 4** — ══════════════════════════════════════════════════════════════════════════  
<sub>1 output block(s) preserved</sub>


In [1]:
# ══════════════════════════════════════════════════════════════════════════
# CELL 8 — THE LAST TWO THINGS
#
#   PART A  Novelty 1 ladder, OFFICIAL SPLIT, all 4 units, MEAN aggregation
#           (replaces the provisional breast/patient figures from Cell 7)
#   PART B  5-FOLD CV re-run with the identical machinery: two-stream +
#           per-BI-RADS layer, floor 0.90, all 4 units, fold-wise fitting
#   PART C  CV vs official side by side — the leak-free check
#
#   No GPU, no training. ~5 minutes.
# ══════════════════════════════════════════════════════════════════════════
import os, glob, json, numpy as np, pandas as pd
from sklearn.metrics import roc_auc_score

D     = "/root/autodl-tmp/CBIS"
FLOOR = 0.90
GRID  = np.round(np.arange(0.01,0.995,0.005),3)
MIN_N, MIN_POS, PASSES = 20, 3, 15
NBOOT, RNG = 2000, np.random.default_rng(7)
stem = lambda p: os.path.splitext(os.path.basename(str(p)))[0]

# ─── shared data ──────────────────────────────────────────────────────────
d = pd.read_csv(os.path.join(D,"unified_folds_mass.csv")); d["_k"]=d["img"].map(stem)
assert d["_k"].is_unique
if "side" not in d.columns:
    fx=pd.read_csv(os.path.join(D,"cbis_mass_fixed.csv")); fx["_k"]=fx["cropped image file path"].map(stem)
    d=d.merge(fx[["_k","side"]].drop_duplicates("_k"),on="_k",how="left")
d["y"]=d["label"].astype(int)
d["a"]=pd.to_numeric(d["assessment"],errors="coerce").fillna(4).astype(int).clip(0,5)
d["breast_key"]=d.patient_id.astype(str)+"_"+d["side"].astype(str)
d["sp"]=np.where(d["official_split"].astype(str).str.lower().str.contains("test"),"test","train")
BASE=d.set_index("_k")[["y","a","lesion_key","breast_key","patient_id","sp","fold"]]
LEVELS=[("IMAGE",None),("LESION","lesion_key"),("BREAST","breast_key"),("PATIENT","patient_id")]

def load(fn):
    f=os.path.join(D,fn)
    if not os.path.exists(f): return None
    m=pd.read_csv(f)
    if "img" not in m.columns: return None
    pc="prob" if "prob" in m.columns else None
    if pc is None:
        c=[x for x in m.columns if m[x].dtype.kind=="f" and m[x].between(0,1).all()]
        if not c: return None
        pc=c[0]
    m["_k"]=m["img"].map(stem)
    m=m[["_k",pc]].rename(columns={pc:"p"}).drop_duplicates("_k")
    m=m[m["_k"].isin(BASE.index)]
    return None if len(m)<300 else m.set_index("_k")["p"]

def roll(series, key):
    """MEAN aggregation — the rule that won on test at every level"""
    t=pd.DataFrame(dict(p=series.values,
                        k=(series.index if key is None else BASE.loc[series.index,key].values),
                        y=BASE.loc[series.index,"y"].values,
                        a=BASE.loc[series.index,"a"].values,
                        pid=BASE.loc[series.index,"patient_id"].values,
                        fold=BASE.loc[series.index,"fold"].values))
    if key is None:
        return t.rename(columns={"k":"key"}).reset_index(drop=True)
    g=t.groupby("k",sort=True)
    o=g.agg(p=("p","mean"),y=("y","max"),a=("a","max"),
            pid=("pid","first"),fold=("fold","first")).reset_index()
    return o.rename(columns={"k":"key"})

# ─── decision-layer machinery (identical to the official run) ─────────────
def _cut(pos,neg,g):
    return (len(pos)-np.searchsorted(pos,g,"left")).astype(float), \
           (len(neg)-np.searchsorted(neg,g,"left")).astype(float)
def fit_global(y,p,fl):
    P,N=int(y.sum()),len(y); tp,fp=_cut(np.sort(p[y==1]),np.sort(p[y==0]),GRID)
    acc,se=(tp+(N-P)-fp)/N,tp/max(P,1); ok=se>=fl-1e-12
    return float(GRID[int(np.argmax(np.where(ok,acc,-1.0)))]) if ok.any() else float(GRID[0])
def _asc(y,p,a,init,fl):
    N,P=len(y),int(y.sum()); cats=np.unique(a)
    tau={int(g):float(init.get(int(g),.5)) for g in cats}
    idx={int(g):np.where(a==g)[0] for g in cats}
    pre={int(g):(np.sort(p[idx[int(g)]][y[idx[int(g)]]==1]),
                 np.sort(p[idx[int(g)]][y[idx[int(g)]]==0])) for g in cats}
    el=[int(g) for g in cats if len(idx[int(g)])>=MIN_N and int(y[idx[int(g)]].sum())>=MIN_POS]
    yh=(p>=np.array([tau[int(g)] for g in a])).astype(int)
    TP=int(((yh==1)&(y==1)).sum()); TN=int(((yh==0)&(y==0)).sum())
    if TP/max(P,1)<fl-1e-12: return tau,-1.0
    for _ in range(PASSES):
        mv=False
        for g in el:
            i=idx[g]; cur=p[i]>=tau[g]
            bTP=TP-int((cur&(y[i]==1)).sum()); bTN=TN-int((~cur&(y[i]==0)).sum())
            pp,nn=pre[g]; tg,fg=_cut(pp,nn,GRID); ng=len(nn)-fg
            acc=(bTP+tg+bTN+ng)/N; se=(bTP+tg)/max(P,1); fe=se>=fl-1e-12
            if not fe.any(): continue
            j=int(np.argmax(np.where(fe,acc,-1.0)))
            if acc[j]>(TP+TN)/N+1e-12:
                tau[g]=float(GRID[j]); TP,TN=int(bTP+tg[j]),int(bTN+ng[j]); mv=True
        if not mv: break
    return tau,(TP+TN)/N
def fit_bir(y,p,a,fl):
    g0=fit_global(y,p,fl); cats=[int(c) for c in np.unique(a)]
    st=[{g:v for g in cats} for v in (g0,0.05,0.15,0.25,0.35,0.45,0.50,0.55,0.65,
                                      max(.01,g0-.1),min(.99,g0+.1))]
    r=np.random.default_rng(0); st+=[{g:float(r.uniform(.05,.8)) for g in cats} for _ in range(6)]
    best,ba=None,-np.inf
    for s in st:
        t,ac=_asc(y,p,a,s,fl)
        if ac>ba+1e-12: best,ba=t,ac
    return (best if best is not None and ba>=0 else {g:g0 for g in cats}), g0
ap=lambda p,a,t,g0:(p>=np.array([t.get(int(g),g0) for g in a])).astype(int)
def M(y,p,yh):
    tp=int(((yh==1)&(y==1)).sum()); fn=int(((yh==0)&(y==1)).sum())
    tn=int(((yh==0)&(y==0)).sum()); fp=int(((yh==1)&(y==0)).sum())
    return dict(auc=roc_auc_score(y,p),acc=(tp+tn)/len(y),sens=tp/max(tp+fn,1),
                spec=tn/max(tn+fp,1),fn=fn)
def ci(y,p,yh,pat,st,nb=NBOOT):
    ps=np.unique(pat); o=[]
    for _ in range(nb):
        ix=np.concatenate([np.where(pat==q)[0] for q in RNG.choice(ps,len(ps),True)])
        if y[ix].min()==y[ix].max(): continue
        o.append(M(y[ix],p[ix],yh[ix])[st])
    return float(np.percentile(o,2.5)),float(np.percentile(o,97.5))

# ══════════════════════════════════════════════════════════════════════════
# PART A — Novelty 1 ladder, OFFICIAL SPLIT, MEAN aggregation
# ══════════════════════════════════════════════════════════════════════════
print("="*92); print("PART A — NOVELTY 1 LADDER, OFFICIAL TEST, MEAN AGGREGATION"); print("="*92)
LADDER=[("baseline  plain DenseNet-121","cv_mass_imageonly_officialsplit_oof.csv"),
        ("N1a       + mask-weighted pooling","cv_mass_officialsplit_oof.csv"),
        ("N1b       + two-stream  [FINAL]","cv_mass_twostream_officialsplit_oof.csv")]
print("  %-36s %-9s %-9s %-9s %s" % ("model","IMAGE","LESION","BREAST","PATIENT"))
LAD={}
for nm,fn in LADDER:
    s=load(fn)
    if s is None: print("  %-36s MISSING (%s)" % (nm,fn)); continue
    s=s[BASE.loc[s.index,"sp"].eq("test").values]
    cells=[]
    for lv,key in LEVELS:
        u=roll(s,key); au=roc_auc_score(u.y,u.p); cells.append(au); LAD[(nm,lv)]=au
    print("  %-36s %s" % (nm," ".join("%.4f   " % c for c in cells)))
print("\n  Novelty 1 total gain, baseline -> N1b  (THESE ARE THE FINAL NUMBERS)")
for lv,_ in LEVELS:
    b,f=LAD.get((LADDER[0][0],lv)),LAD.get((LADDER[2][0],lv))
    if b and f: print("    %-8s  %.4f -> %.4f   %+.4f AUC" % (lv,b,f,f-b))

# ══════════════════════════════════════════════════════════════════════════
# PART B — 5-FOLD CV, identical machinery
# ══════════════════════════════════════════════════════════════════════════
print("\n"+"="*92); print("PART B — 5-FOLD CV (patient-grouped), floor %.2f, MEAN aggregation" % FLOOR)
print("  thresholds fitted on 4 folds, frozen, applied to the held-out fold; repeated 5x")
print("="*92)

print("\n  CV out-of-fold AUC of every usable prediction file (image level)")
for f in sorted(glob.glob(os.path.join(D,"cv_mass_*_oof.csv"))+glob.glob(os.path.join(D,"cv_mass_*s??.csv"))):
    if "official" in os.path.basename(f): continue
    s=load(os.path.basename(f))
    if s is None or len(s)<1500: continue
    print("    %-44s n=%d  AUC %.4f"
          % (os.path.basename(f),len(s),roc_auc_score(BASE.loc[s.index,"y"].values,s.values)))

cvs=load("cv_mass_twostream_oof.csv")
assert cvs is not None, "cv_mass_twostream_oof.csv not usable"
print("\n  %-8s %-5s %-19s %-18s %-6s %-6s %-8s %s"
      % ("level","n","AUC [95% CI]","acc [95% CI]","sens","spec","missed","vs 1 threshold"))
CVOUT={}
for lv,key in LEVELS:
    u=roll(cvs,key); u=u[u.fold.notna()].reset_index(drop=True)
    y,p,a,f,pid = (u.y.values.astype(int),u.p.values,u.a.values.astype(int),
                   u.fold.values.astype(int),u.pid.values)
    yh1=np.zeros(len(u),int); yh2=np.zeros(len(u),int)
    for k in sorted(np.unique(f)):
        tr,te=f!=k,f==k
        g1=fit_global(y[tr],p[tr],FLOOR); t2,g2=fit_bir(y[tr],p[tr],a[tr],FLOOR)
        yh1[te]=(p[te]>=g1).astype(int); yh2[te]=ap(p[te],a[te],t2,g2)
    m1,m2=M(y,p,yh1),M(y,p,yh2)
    la,ha=ci(y,p,yh2,pid,"auc"); lc,hc=ci(y,p,yh2,pid,"acc")
    CVOUT[lv]=dict(m2,auc_ci=[la,ha],acc_ci=[lc,hc],acc_one=m1["acc"])
    print("  %-8s %-5d %.4f[%.3f-%.3f] %.1f%%[%.1f-%.1f] %.3f  %.3f  %3d/%-4d %.1f%% (%+.2f)"
          % (lv,len(y),m2["auc"],la,ha,100*m2["acc"],100*lc,100*hc,m2["sens"],m2["spec"],
             m2["fn"],int(y.sum()),100*m1["acc"],100*(m2["acc"]-m1["acc"])))

# ══════════════════════════════════════════════════════════════════════════
# PART C — CV vs OFFICIAL
# ══════════════════════════════════════════════════════════════════════════
OFF={"IMAGE":(0.8769,0.810),"LESION":(0.9043,0.857),"BREAST":(0.9016,0.848),"PATIENT":(0.9041,0.851)}
print("\n"+"="*92); print("PART C — 5-FOLD CV vs OFFICIAL SPLIT  (leak-free check)"); print("="*92)
print("  %-8s %-22s %-22s %s" % ("level","5-fold CV","official test","CV minus official"))
for lv,_ in LEVELS:
    c=CVOUT[lv]; o=OFF[lv]
    print("  %-8s AUC %.4f  acc %.1f%%   AUC %.4f  acc %.1f%%   %+.4f AUC, %+.2f pts"
          % (lv,c["auc"],100*c["acc"],o[0],100*o[1],c["auc"]-o[0],100*(c["acc"]-o[1])))
print("\n  A CV result at or BELOW the official-split result means your cross-validation")
print("  does not inflate. A large positive gap would indicate leakage across folds.")

with open(os.path.join(D,"FINAL_cv_and_ladder.json"),"w") as fh:
    json.dump(dict(floor=FLOOR,aggregation="mean",
                   ladder={"%s|%s"%k:v for k,v in LAD.items()},cv=CVOUT),fh,indent=2,default=float)
print("\nsaved FINAL_cv_and_ladder.json")

PART A — NOVELTY 1 LADDER, OFFICIAL TEST, MEAN AGGREGATION
  model                                IMAGE     LESION    BREAST    PATIENT
  baseline  plain DenseNet-121         0.7994    0.8092    0.7970    0.7960   
  N1a       + mask-weighted pooling    0.8480    0.8715    0.8647    0.8651   
  N1b       + two-stream  [FINAL]      0.8769    0.9043    0.9016    0.9041   

  Novelty 1 total gain, baseline -> N1b  (THESE ARE THE FINAL NUMBERS)
    IMAGE     0.7994 -> 0.8769   +0.0775 AUC
    LESION    0.8092 -> 0.9043   +0.0951 AUC
    BREAST    0.7970 -> 0.9016   +0.1046 AUC
    PATIENT   0.7960 -> 0.9041   +0.1080 AUC

PART B — 5-FOLD CV (patient-grouped), floor 0.90, MEAN aggregation
  thresholds fitted on 4 folds, frozen, applied to the held-out fold; repeated 5x

  CV out-of-fold AUC of every usable prediction file (image level)
    cv_mass_blind_v4_oof.csv                     n=1696  AUC 0.8190
    cv_mass_convnext_tiny_oof.csv                n=1696  AUC 0.8566
    cv_mass_dualpath_